# Week 65, Day 3: Fast data structures

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w065d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import heapq, bisect
from collections import Counter, deque

scores = [52, 91, 77, 68, 99, 85]
print(heapq.nlargest(2, scores))
ordered = sorted(scores)
print(bisect.bisect_left(ordered, 70), len(ordered) - bisect.bisect_left(ordered, 80))
print(Counter("mississippi").most_common(2))
recent = deque(maxlen=3)
for s in scores:
    recent.append(s)
print(list(recent))

In [ ]:
import time
from collections import deque

def drain_list(n):
    q = list(range(n))
    while q:
        q.pop(0)              # shifts every remaining item

def drain_deque(n):
    q = deque(range(n))
    while q:
        q.popleft()           # O(1)

for drain in (drain_list, drain_deque):
    start = time.perf_counter()
    drain(30000)
    print(drain.__name__, "done", time.perf_counter() - start >= 0)

emails = [f"user{i}@example.com" for i in range(10000)]
known = set(emails)                       # convert once
print(sum(1 for i in range(0, 20000, 7) if f"user{i}@example.com" in known))

In [ ]:
import heapq

def top_k_stream(stream, k):
    heap = []
    for x in stream:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:
            heapq.heapreplace(heap, x)       # pop the smallest and push x, in one step
    return sorted(heap, reverse=True)

print(top_k_stream((n * 7919 % 1000 for n in range(10000)), 3))

tasks = []
for priority, name in [(3, "email"), (1, "fire"), (2, "lunch")]:
    heapq.heappush(tasks, (priority, name))
print([heapq.heappop(tasks)[1] for _ in range(3)])

print(list(heapq.merge([1, 4, 9], [2, 3, 10], [5])))
print(heapq.nlargest(2, [("ada", 90), ("ben", 72), ("cy", 95)], key=lambda p: p[1]))

In [ ]:
import bisect

times = [905, 912, 930, 1001, 1015, 1015, 1102, 1230]

def count_between(sorted_items, lo, hi):
    return bisect.bisect_right(sorted_items, hi) - bisect.bisect_left(sorted_items, lo)

print(count_between(times, 1000, 1100), count_between(times, 1015, 1015))

def next_after(sorted_items, x):
    i = bisect.bisect_right(sorted_items, x)
    return sorted_items[i] if i < len(sorted_items) else None

print(next_after(times, 1015), next_after(times, 1300))

bisect.insort(times, 1000)
print(times[:5])

grades = [(0, "fail"), (40, "pass"), (60, "merit"), (70, "distinction")]
def grade(score):
    i = bisect.bisect_right([g[0] for g in grades], score) - 1
    return grades[i][1]
print([grade(s) for s in (12, 40, 65, 99)])

In [ ]:
from collections import Counter, defaultdict

words = "the cat and the hat and the bat".split()
counts = Counter(words)
print(counts.most_common(2), counts["dog"])

orders = [("ada", "tea"), ("ben", "jam"), ("ada", "jam"), ("ada", "tea")]
by_customer = defaultdict(list)
for customer, item in orders:
    by_customer[customer].append(item)
print(dict(by_customer))

unique_items = defaultdict(set)
for customer, item in orders:
    unique_items[customer].add(item)
print({k: sorted(v) for k, v in unique_items.items()})

people = [{"id": 7, "name": "Ada"}, {"id": 3, "name": "Ben"}]
by_id = {p["id"]: p for p in people}
print(by_id[3]["name"])

In [ ]:
import sys
from array import array

values = [float(i) for i in range(10000)]
packed = array("d", values)
list_bytes = sys.getsizeof(values) + sum(sys.getsizeof(v) for v in values)
print(packed[10], len(packed), list_bytes > 2 * sys.getsizeof(packed))

class Plain:
    def __init__(self, x, y):
        self.x, self.y = x, y

class Slotted:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x, self.y = x, y

p, s = Plain(1, 2), Slotted(1, 2)
print(sys.getsizeof(p) + sys.getsizeof(p.__dict__) > sys.getsizeof(s), hasattr(s, "__dict__"))

> **Key idea.** Choose structures by the operations you do most: lists for indexing and appending, dicts and sets for O(1) lookups (convert once), deques for both ends, heapq for repeated smallest-item access and top-k (O(k) memory for streams, negate keys for a max-heap), and sorted lists with bisect for ranges, ranks and next-item queries (bisect_right(hi) - bisect_left(lo) counts a range). Counter counts, defaultdict groups, and dicts built once serve as indexes. Keep running results instead of recomputing. Store large homogeneous data compactly with array, bytes, __slots__ or generators.

### Exercise: The best of a stream

Write `top_k(stream, k, key=None)`: the `k` largest items of an iterable, largest first, keeping **at most `k`** items in a heap at any time (so it works on streams too long to store). With `key`, compare by `key(item)`; among equal keys, the item seen **first** comes first. Then `merged(*sorted_lists)`: one sorted list from several already-sorted lists, using `heapq.merge`.

In [ ]:
import heapq

def top_k(stream, k, key=None):
    ___

def merged(*sorted_lists):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_topk():\n    \"heapq\"\n    assert top_k(iter([5, 1, 9, 3, 7]), 3) == [9, 7, 5] and top_k([], 3) == [] and top_k([4], 0) == []\n    people = [(\"ada\", 90), (\"ben\", 95), (\"cy\", 90), (\"di\", 70)]\n    assert top_k(people, 3, key=lambda p: p[1]) == [(\"ben\", 95), (\"ada\", 90), (\"cy\", 90)]\n    def huge():\n        for i in range(100003):\n            yield (i * 7919) % 100003\n    assert top_k(huge(), 2) == [100002, 100001]\n    assert merged([1, 5], [2, 3, 9], []) == [1, 2, 3, 5, 9]\n    assert \"heapq\" in source")

### Exercise: Counting in ranges

Write a class `Timeline` that keeps event times in a sorted list using `bisect`: `add(t)` (keeping it sorted), `count(lo, hi)` (events with `lo <= t <= hi`, by binary search), `before(t)` (how many events are strictly earlier than `t`), and `next_after(t)` (the first event **strictly** later than `t`, or `None`). Don't sort the whole list after each `add`.

In [ ]:
import bisect

class Timeline:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ranges():\n    \"bisect\"\n    tl = Timeline()\n    for t in [30, 10, 20, 20, 50]:\n        tl.add(t)\n    assert tl.count(15, 30) == 3 and tl.count(20, 20) == 2 and tl.count(60, 90) == 0\n    assert tl.before(20) == 1 and tl.before(100) == 5 and tl.before(0) == 0\n    assert tl.next_after(20) == 30 and tl.next_after(50) is None and tl.next_after(5) == 10\n    import re\n    assert not re.search(r\"(?<!in)sort\\(\", source) and \"sorted(\" not in source")

### Exercise: Grouping and counting

Given sales as `(region, product, amount)` tuples, write `by_region(sales)`: a plain dict from region to the list of its sales, in order (use `defaultdict(list)`); `best_sellers(sales, k)`: the `k` products with the largest **total amount**, as `(product, total)` pairs, largest first, ties in order of first appearance (use a `Counter`); and `regions_per_product(sales)`: a dict from product to the sorted list of regions it sold in.

In [ ]:
from collections import Counter, defaultdict

def by_region(sales):
    ___

def best_sellers(sales, k):
    ___

def regions_per_product(sales):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_group():\n    \"Counter and defaultdict\"\n    sales = [(\"north\", \"tea\", 5), (\"south\", \"jam\", 8), (\"north\", \"jam\", 2), (\"east\", \"tea\", 5), (\"south\", \"milk\", 10)]\n    assert by_region(sales) == {\"north\": [(\"north\", \"tea\", 5), (\"north\", \"jam\", 2)], \"south\": [(\"south\", \"jam\", 8), (\"south\", \"milk\", 10)], \"east\": [(\"east\", \"tea\", 5)]}\n    assert type(by_region(sales)) is dict\n    assert best_sellers(sales, 2) == [(\"tea\", 10), (\"jam\", 10)] and best_sellers(sales, 5)[-1] == (\"milk\", 10)\n    assert best_sellers(sales, 3) == [(\"tea\", 10), (\"jam\", 10), (\"milk\", 10)]\n    assert regions_per_product(sales) == {\"tea\": [\"east\", \"north\"], \"jam\": [\"north\", \"south\"], \"milk\": [\"south\"]}")

### Exercise: Running results

Write `MovingAverage(window)`: `add(x)` adds a value and returns the average of the last `window` values (or of all values so far, if fewer). Keep a `deque(maxlen=window)` and a **running total** updated in O(1) per call: the tests count how many values you sum. Then `running_max(values)`: a list where each entry is the largest value so far.

In [ ]:
from collections import deque

class MovingAverage:
    def __init__(self, window):
        ___

def running_max(values):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_running():\n    \"O(1) updates\"\n    m = MovingAverage(3)\n    assert [m.add(x) for x in (3, 6, 9, 12)] == [3.0, 4.5, 6.0, 9.0]\n    big = MovingAverage(1000)\n    for i in range(20000):\n        last = big.add(i)\n    assert last == 19499.5\n    assert \"sum(\" not in source\n    assert running_max([3, 1, 4, 1, 5]) == [3, 3, 4, 4, 5] and running_max([]) == []")

### Exercise: Storing it smaller

Write `pack(values)`: an `array.array` of 8-byte floats (`"d"`) holding the values, and `unpack(packed)`: a plain list of floats again. Then `bytes_used(obj)`: `sys.getsizeof(obj)` plus, for a list, the sizes of its items (each distinct item counted once by `id`). Finally `Reading`, a small class with `sensor`, `time` and `value` attributes that uses `__slots__`, so it has no `__dict__`.

In [ ]:
import sys
from array import array

def pack(values):
    ___

def unpack(packed):
    ___

def bytes_used(obj):
    ___

class Reading:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_compact():\n    \"compact storage\"\n    import sys\n    values = [i / 3 for i in range(5000)]\n    packed = pack(values)\n    assert type(packed).__name__ == \"array\" and packed.typecode == \"d\" and unpack(packed) == values\n    assert bytes_used(values) > 2 * bytes_used(packed)\n    shared = 2.5\n    assert bytes_used([shared, shared]) == sys.getsizeof([shared, shared]) + sys.getsizeof(shared)\n    r = Reading(\"s1\", 10, 21.5)\n    assert (r.sensor, r.time, r.value) == (\"s1\", 10, 21.5) and not hasattr(r, \"__dict__\")\n    try:\n        r.colour = \"red\"\n        assert False\n    except AttributeError:\n        pass")

### Bug hunt: A min-heap used as a max-heap

`heapq` always pops the **smallest** item, so this "highest bid first" auction serves the lowest bid and prints `[5, 12, 30]`. Push the bids negated (and negate them again when popping) so it prints `[30, 12, 5]`.

In [ ]:
import heapq

bids = []
for amount in (12, 30, 5):
    heapq.heappush(bids, amount)
print([heapq.heappop(bids) for _ in range(3)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a max-heap\"\n    expect_output(\"[30, 12, 5]\")")

### Bug hunt: One short in the range

To count scores from 60 to 80 **inclusive**, the code uses `bisect_left` for the top end, which stops before any scores **equal** to 80, so this prints `2`. Use `bisect_right` for the upper bound so it prints `3`.

In [ ]:
import bisect

scores = sorted([55, 60, 72, 80, 91])
print(bisect.bisect_left(scores, 80) - bisect.bisect_left(scores, 60))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"inclusive ranges\"\n    expect_output(\"3\")")

### Bug hunt: Grouping into None

`list.append` returns `None`, so `groups[key] = groups[key].append(item)` replaces each group's list with `None`, and the second item for a key crashes with `AttributeError`. Just call `append` on the group (the `defaultdict` creates the list) so this prints `{'a': [1, 3], 'b': [2]}`.

In [ ]:
from collections import defaultdict

groups = defaultdict(list)
for key, item in [("a", 1), ("b", 2), ("a", 3)]:
    groups[key] = groups[key].append(item)
print(dict(groups))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"append in place\"\n    expect_output(\"{'a': [1, 3], 'b': [2]}\")")

### Puzzle: Top k of a stream

Arrange the lines of a function that keeps the k largest numbers of a stream in a heap of size k. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the structure for a queue you add to at the back and take from the front
q = ___   # bisect.bisect_left([10, 20, 20, 30], 20)
r = ___   # bisect.bisect_right([10, 20, 20, 30], 20)
s = ___   # Counter("abca").most_common(1)
t = ___   # True or False: an array('d') of a million floats is smaller than a list of the same floats

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"deque\" and q == 1 and r == 3, \"p, q and r\"\n    assert s == [(\"a\", 2)] and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a fast leaderboard

Write `Leaderboard` for a game with frequent score updates and frequent queries. Keep a dict from player to score **and** a sorted list of `(-score, name)` entries maintained with `bisect`, so that queries never sort:

- `add(name, points)`: add points to the player's score (new players start at 0; points may be negative);
- `score(name)` (`KeyError` for unknown players) and `remove(name)`;
- `top(k)`: the best `k` as `(name, score)` pairs: highest score first, ties alphabetical;
- `rank(name)`: the player's 1-based position in that order;
- `around(name, d)`: the players up to `d` places above and below (including the player), in order;
- `len(board)`.

Each update must find and remove the player's old entry with a binary search (not `list.remove`, which scans).

In [ ]:
import bisect

class Leaderboard:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_board():\n    \"a leaderboard\"\n    b = Leaderboard()\n    for name, pts in [(\"ada\", 50), (\"ben\", 70), (\"cy\", 50), (\"di\", 20), (\"ada\", 30)]:\n        b.add(name, pts)\n    assert b.top(3) == [(\"ada\", 80), (\"ben\", 70), (\"cy\", 50)] and len(b) == 4\n    assert b.rank(\"cy\") == 3 and b.rank(\"ada\") == 1 and b.score(\"di\") == 20\n    assert b.around(\"cy\", 1) == [(\"ben\", 70), (\"cy\", 50), (\"di\", 20)] and b.around(\"ada\", 2)[0] == (\"ada\", 80)\n    b.add(\"ben\", -25)\n    assert b.top(4) == [(\"ada\", 80), (\"cy\", 50), (\"ben\", 45), (\"di\", 20)]\n    b.remove(\"ada\")\n    assert b.rank(\"cy\") == 1 and len(b) == 3\n    try:\n        b.score(\"ada\")\n        assert False\n    except KeyError:\n        pass\n    assert \".remove(\" not in source.replace(\"def remove(\", \"\") and \"sorted(\" not in source\n    import re\n    assert not re.search(r\"(?<!in)sort\\(\", source)\n\ndef test_against_brute_force():\n    \"matches a simple version\"\n    import random\n    random.seed(11)\n    b, scores = Leaderboard(), {}\n    names = [f\"p{i}\" for i in range(30)]\n    for _ in range(2000):\n        name, pts = random.choice(names), random.randint(-20, 50)\n        b.add(name, pts)\n        scores[name] = scores.get(name, 0) + pts\n    expected = sorted(scores.items(), key=lambda p: (-p[1], p[0]))\n    assert b.top(30) == expected\n    assert all(b.rank(n) == i + 1 for i, (n, _) in enumerate(expected))")

## Warm-up

Five structures, five jobs: the two best scores without sorting everything, how many scores fall below 70 or at least 80 by binary search, the most common letters, and a window of the last three values that maintains itself.

## Choosing by operation

Start from the operations your program does most, then pick the structure that makes those cheap (average costs, n items):

| structure | fast | slow |
|---|---|---|
| `list` | index, append, pop from the end, sort, iterate | `in`, insert/remove at the front or middle: O(n) |
| `dict` | look up, add, remove by key: O(1); keeps insertion order | finding a key by its value: O(n) |
| `set` | `in`, add, remove: O(1); union, intersection | order, indexing |
| `collections.deque` | append and pop at **both** ends: O(1) | indexing in the middle: O(n) |
| `heapq` on a list | push, pop the smallest: O(log n); see the smallest: O(1) | search, sorted iteration |
| sorted list + `bisect` | binary search, ranges, rank: O(log n) | insert or remove: O(n) shifting (but fast, in C) |
| `Counter`, `defaultdict` | counting and grouping with no special cases | (dict costs) |
| `array`, `bytes` | compact storage of numbers or bytes | mixed types |

Two habits: **turn a list into a set or dict once** when you'll do many lookups, and **keep data in the shape you query it in** (an index by id if you look things up by id).

## Heaps with heapq

A **heap** keeps the smallest item at index 0 of a list, with pushes and pops in O(log n) (Week 48). `heapq` works on a plain list:

- `heapq.heappush(h, x)`, `heapq.heappop(h)` (smallest), `h[0]` (peek), `heapq.heapify(list)` (O(n));
- `heapq.nlargest(k, items, key=...)` and `nsmallest`: the top k in O(n log k), without sorting everything;
- `heapq.merge(*sorted_iterables)`: merge already-sorted inputs lazily (great for huge sorted files).

For a **max-heap**, push negated keys (`-score`) or tuples like `(-score, name)`. For a **top-k of a stream**, keep a min-heap of size k: if a new item beats the smallest of the k, replace it (`heapq.heappushpop` or `heapreplace`). Memory stays O(k) however long the stream is.

## Sorted data with bisect

If you query data by order (ranges, ranks, "the next one after x"), keep it **sorted** and use binary search from `bisect`:

- `bisect.bisect_left(a, x)`: the first position where `x` could go (the number of items **less than** `x`);
- `bisect.bisect_right(a, x)`: the position after any items equal to `x` (the number **less than or equal to** `x`);
- `bisect.insort(a, x)`: insert keeping the list sorted (the search is O(log n); the insertion shifts items, O(n), but very fast in C);
- since Python 3.10, `key=` lets you search lists of records.

Counting the items in a range `[lo, hi]` is `bisect_right(a, hi) - bisect_left(a, lo)`: two binary searches instead of a scan. For data that changes a lot and is queried by order, the `sortedcontainers` library (free, pure Python) provides a sorted list with fast insertion; in the standard library, sorted list plus `bisect` is the tool.

## Counting, grouping and indexing

Many programs spend their time counting things or finding records by a field. The `collections` module (Week 20) has the right tools:

- **`Counter`**: `Counter(iterable)` counts in C; `.most_common(k)`, arithmetic between counters (`c1 + c2`, `c1 - c2`), `.update(more)`. Ties in `most_common` keep first-seen order.
- **`defaultdict(list)`** for grouping: `groups[key].append(item)` with no "is the key there yet?" check; `defaultdict(int)` for counting with custom keys; `defaultdict(set)` for unique values per key.
- **A dict as an index**: build `{record["id"]: record for record in records}` once, then every lookup is O(1). Build several indexes if you query by several fields.
- **`dict.setdefault` and `dict.get`** for the occasional default without a special dict.

## Compact storage

Memory costs time as well as space: smaller data means fewer cache misses, less garbage collection, and fitting in RAM at all. A Python `int` or `float` is a full object (around 24 to 32 bytes on a 64-bit machine), and a list stores an 8-byte **pointer** to each (Week 64). For large homogeneous data:

- **`array.array(typecode, ...)`** stores raw machine numbers: `"d"` for 8-byte floats, `"i"` for 4-byte ints, `"b"` for single bytes. It's a sequence like a list (indexing, slicing, `append`), but typically 2 to 4 times smaller than a list of numbers (more on 64-bit machines, where pointers and objects are bigger).
- **`bytes` / `bytearray`** for byte data, one byte per value.
- **`__slots__`** (Week 31) removes each instance's `__dict__`, saving tens of bytes per object when you have millions of small objects. Tuples and named tuples are also smaller than objects with dicts.
- **Generators** (Week 59) don't store the data at all.
- For serious numerical work, NumPy arrays (Friday) combine compact storage with fast operations.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at the table in "Choosing by operation" and, for a program you've written, list the two operations it does most often. Is it using the structure that makes those cheap? Tomorrow: caching, the general cure for repeated work.